# TFM Tennis CV — Pipeline completo (Kaggle)

**Antes de ejecutar**, añade estos dos Datasets en la pestaña *Data* del notebook:

| Dataset Kaggle | Contenido | Ruta en Kaggle |
|---|---|---|
| `tfm-tennis-dataset` | `Dataset/game*` (frames + Label.csv) | `/kaggle/input/tfm-tennis-dataset/` |
| `tfm-tennis-model` | `tracknet_best.pth` | `/kaggle/input/tfm-tennis-model/` |

El código del proyecto se sube como tercer Dataset o se clona desde GitHub (celda 2).

## 1. Configuración de rutas

In [ ]:
import os
from pathlib import Path

# Rutas de los Datasets añadidos en la pestaña Data.
# Ajusta los nombres de carpeta si los subiste con otro slug.
DATASET_ROOT = '/kaggle/input/tfm-tennis-dataset/Dataset'
MODEL_SRC    = '/kaggle/input/tfm-tennis-model/tracknet_best.pth'

# Outputs en /kaggle/working (persistente durante la sesión; descárgalo al acabar).
OUTPUTS_ROOT = '/kaggle/working/outputs'
REPO_DIR     = '/kaggle/working/ProyectoTFM'

# Copiar el checkpoint al sitio donde config.py lo espera.
models_dir = Path(OUTPUTS_ROOT) / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy(MODEL_SRC, models_dir / 'tracknet_best.pth')

# Variables de entorno para config.py.
os.environ['TFM_PROJECT_ROOT'] = REPO_DIR
os.environ['TFM_DATASET_ROOT'] = DATASET_ROOT
os.environ['TFM_OUTPUTS_ROOT'] = OUTPUTS_ROOT

print('Dataset  :', DATASET_ROOT)
print('Outputs  :', OUTPUTS_ROOT)
print('Modelo   :', models_dir / 'tracknet_best.pth')

## 2. Clonar / actualizar el repositorio

In [ ]:
import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print('Repo ya existe, actualizando...')
    !cd "{REPO_DIR}" && git pull
else:
    REPO_URL = 'https://github.com/TU_USUARIO/ProyectoTFM.git'
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)
print('Directorio de trabajo:', os.getcwd())

## 3. Instalar dependencias

In [ ]:
# Kaggle ya trae torch/torchvision con CUDA.
!pip install -q ultralytics lap openpyxl

import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Verificar estructura

In [ ]:
from pathlib import Path

checks = {
    'Dataset':           Path(DATASET_ROOT),
    'tracknet_best.pth': Path(OUTPUTS_ROOT) / 'models' / 'tracknet_best.pth',
    'config.py':         Path(REPO_DIR) / 'config.py',
    'main.py':           Path(REPO_DIR) / 'main.py',
}

all_ok = True
for nombre, ruta in checks.items():
    ok = ruta.exists()
    print(f"  {'OK' if ok else 'FALTA'}  {nombre}: {ruta}")
    all_ok = all_ok and ok

if all_ok:
    print('\nTodo listo.')
else:
    print('\nFaltan rutas. Revisa la celda de configuracion.')

## 5. Ejecutar el pipeline (un game)

In [ ]:
GAME = 'game1'  # <-- cambia aquí

!python main.py \
    --game-path "{DATASET_ROOT}/{GAME}" \
    --output-dir "{OUTPUTS_ROOT}/{GAME}" \
    --log-level INFO

## 6. (Opcional) Procesar todos los games

In [ ]:
from pathlib import Path

games = sorted(Path(DATASET_ROOT).glob('game*'))
print(f'Games encontrados: {[g.name for g in games]}')

for game_path in games:
    print(f'\n=== Procesando {game_path.name} ===')
    out = Path(OUTPUTS_ROOT) / game_path.name
    !python main.py \
        --game-path "{game_path}" \
        --output-dir "{out}" \
        --log-level INFO

## 7. Descargar outputs

Al terminar, los outputs están en `/kaggle/working/outputs/`.  
Descárgalos desde la pestaña *Output* del notebook, o comprime y descarga:

In [ ]:
!cd /kaggle/working && zip -r outputs.zip outputs/
print('Descarga outputs.zip desde la pestaña Output del notebook.')